# TFM Tenerife — Ingesta de la Red de Transporte (GTFS TITSA y Metropolitano)

Este notebook descarga los ficheros GTFS de TITSA (guaguas) y de Metropolitano de Tenerife (tranvía), construye las paradas y las líneas de cada red, y las sube al esquema `silver` de Azure Database for PostgreSQL (ya reproyectadas a EPSG:32628). El dato en crudo vive en el Data Lake (contenedor `bronze_route`), este notebook no lo toca.


## Paso 1 — Instalar dependencias

In [ ]:
#!pip install geopandas sqlalchemy psycopg2-binary geoalchemy2 shapely requests python-dotenv



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Paso 2 — Conectar con Azure Database for PostgreSQL

Misma conexión que el notebook 8: variable `AZURE_DB_URL` leída del `.env` local.


In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(r"C:\Users\mario\Desktop\TFM\.env")

conn_str = os.environ['AZURE_DB_URL']
engine = create_engine(conn_str, pool_pre_ping=True, pool_recycle=1800)

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. Version de PostGIS:', version)


Conectado. Version de PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## Paso 3 — Qué es un GTFS y de dónde lo sacamos

Un GTFS no es un único fichero: es un `.zip` que contiene varias tablas en texto plano (formato CSV). Las que nos interesan para la topología espacial son:

- `stops.txt` — las paradas, con su latitud y longitud (puntos).
- `shapes.txt` — el trazado real de cada línea, punto a punto (líneas).
- `routes.txt` y `trips.txt` — para poder ponerle nombre a cada línea (ej. 'Línea 015').

En vez de guardar la URL fija del ZIP, guardamos el `package_id` de CKAN: la URL de descarga real y la fecha de última modificación se consultan en el Paso 4b vía la API de datos.tenerife.es, así no dependemos de una URL que puede cambiar de `resource_id` si el operador resube el fichero, y podemos saltarnos la descarga si el feed no ha cambiado.


In [3]:
FUENTES_GTFS = {
    'titsa': {
        'package_id': '36c2e26f-0d18-4b5a-b214-1636168e0765',
        'operador': 'TITSA',
        'modo': 'guagua',
    },
    'metropolitano': {
        'package_id': '4b83e018-37d9-40a6-b6d1-1df2b91c8117',
        'operador': 'Metropolitano de Tenerife',
        'modo': 'tranvia',
    },
}


## Paso 4 — Funciones para descargar y leer el GTFS

Un GTFS se descarga como un `.zip` en memoria y se lee directamente de ahí, sin necesidad de guardarlo en disco.

In [4]:
import io
import zipfile
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString


def descargar_gtfs(url):
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    return zipfile.ZipFile(io.BytesIO(resp.content))


def leer_tabla(zf, nombre_fichero):
    with zf.open(nombre_fichero) as f:
        return pd.read_csv(f)

## Paso 4b — Consultar la API de CKAN en vez de bajar el ZIP a ciegas

Antes de descargar nada, preguntamos a la API `package_show` por el recurso actual: URL real de descarga, fecha de última modificación (`last_modified`) y tamaño. Con eso decidimos si merece la pena descargar y reprocesar.


In [5]:
CKAN_API = 'https://datos.tenerife.es/ckan/api/action/package_show'

def obtener_recurso_gtfs(package_id):
    """Consulta la API CKAN y devuelve la info del recurso ZIP del feed GTFS."""
    resp = requests.get(CKAN_API, params={'id': package_id}, timeout=30)
    resp.raise_for_status()
    recurso = resp.json()['result']['resources'][0]
    return {
        'url': recurso['url'],
        'last_modified': recurso['last_modified'],
        'size': recurso['size'],
    }


## Paso 4c — Tabla de control de cargas (por operador)

Igual que en el notebook 8, guardamos en la propia base de datos la fecha de la última carga por operador. Usamos una tabla separada, `gtfs_control_carga_geo`, porque este notebook construye capas espaciales (`gtfs_paradas`, `gtfs_rutas`) distintas de las tablas relacionales del notebook 8 -- son procesos independientes que pueden ejecutarse en momentos distintos.


In [6]:
def _tabla_existe(nombre, esquema):
    with engine.connect() as conn:
        return conn.execute(
            text('SELECT to_regclass(:t) IS NOT NULL'), {'t': f'{esquema}.{nombre}'}
        ).scalar()

def asegurar_tabla_control_geo():
    with engine.begin() as conn:
        conn.execute(text('''
            CREATE TABLE IF NOT EXISTS silver.gtfs_control_carga_geo (
                operador TEXT PRIMARY KEY,
                last_modified TIMESTAMPTZ
            );
        '''))

def feed_ha_cambiado_geo(operador, last_modified_actual):
    asegurar_tabla_control_geo()
    with engine.connect() as conn:
        fila = conn.execute(
            text('SELECT last_modified FROM silver.gtfs_control_carga_geo WHERE operador = :op'),
            {'op': operador}
        ).fetchone()
    if fila is None:
        return True
    return str(fila[0]) != str(last_modified_actual)

def actualizar_control_carga_geo(operador, last_modified_actual):
    with engine.begin() as conn:
        conn.execute(text('''
            INSERT INTO silver.gtfs_control_carga_geo (operador, last_modified)
            VALUES (:op, :lm)
            ON CONFLICT (operador) DO UPDATE SET last_modified = EXCLUDED.last_modified;
        '''), {'op': operador, 'lm': last_modified_actual})


## Paso 5 — Construir las paradas (puntos)

Cada fila de `stops.txt` se convierte en un punto (`Point`) usando su longitud y latitud.

In [7]:
def construir_paradas(zf, operador, modo):
    stops = leer_tabla(zf, 'stops.txt')
    stops = stops.dropna(subset=['stop_lat', 'stop_lon'])
    geometry = [Point(xy) for xy in zip(stops['stop_lon'], stops['stop_lat'])]
    gdf = gpd.GeoDataFrame(stops, geometry=geometry, crs='EPSG:4326')
    gdf['operador'] = operador
    gdf['modo'] = modo
    columnas = ['stop_id', 'stop_name', 'operador', 'modo', 'geometry']
    return gdf[[c for c in columnas if c in gdf.columns]]

## Paso 6 — Construir las líneas de cada ruta

Cada `shape_id` de `shapes.txt` es una secuencia de puntos que, unidos en orden (`shape_pt_sequence`), forman una línea (`LineString`). Le añadimos el nombre de la ruta cruzando con `trips.txt` y `routes.txt`.

In [8]:
def construir_rutas(zf, operador, modo):
    archivos = zf.namelist()
    columnas = ['shape_id', 'route_short_name', 'route_long_name', 'operador', 'modo', 'geometry']

    if 'shapes.txt' not in archivos:
        print('  aviso:', operador, 'no tiene shapes.txt, no se construyen lineas de ruta')
        return gpd.GeoDataFrame(columns=columnas, geometry='geometry', crs='EPSG:4326')

    shapes = leer_tabla(zf, 'shapes.txt')
    trips = leer_tabla(zf, 'trips.txt')
    routes = leer_tabla(zf, 'routes.txt')

    trips_unicos = trips.drop_duplicates(subset='shape_id')[['shape_id', 'route_id']]
    info_rutas = trips_unicos.merge(routes, on='route_id', how='left')

    shapes_sorted = shapes.sort_values(['shape_id', 'shape_pt_sequence'])
    lineas = []
    for shape_id, grupo in shapes_sorted.groupby('shape_id'):
        linea = LineString(zip(grupo['shape_pt_lon'], grupo['shape_pt_lat']))
        lineas.append({'shape_id': shape_id, 'geometry': linea})

    gdf = gpd.GeoDataFrame(lineas, crs='EPSG:4326')
    gdf = gdf.merge(info_rutas, on='shape_id', how='left')
    gdf['operador'] = operador
    gdf['modo'] = modo
    return gdf[[c for c in columnas if c in gdf.columns]]

## Paso 7 — Función para subir una capa a silver

Sube los datos de un operador a la vez, ya reproyectados a EPSG:32628: si la tabla `silver.<nombre>` no existe la crea con su índice GIST; si ya existe, borra solo las filas de ese operador (`WHERE operador = ...`) y añade las nuevas, sin tocar los datos del otro operador.


In [9]:
def subir_capa(gdf, nombre, operador):
    """Sube los datos de UN operador a la tabla silver.<nombre> (reproyectada a
    EPSG:32628). Si la tabla no existe, la crea; si existe, borra solo las filas
    de ese operador y añade las suyas, sin tocar los datos del otro operador."""
    gdf_proc = gdf.to_crs(epsg=32628)
    existe = _tabla_existe(nombre, 'silver')

    if existe:
        with engine.begin() as conn:
            conn.execute(
                text(f'DELETE FROM silver.{nombre} WHERE operador = :op'),
                {'op': operador}
            )

    gdf_proc.to_postgis(
        nombre, engine, schema='silver',
        if_exists='append' if existe else 'replace', index=False
    )
    with engine.begin() as conn:
        conn.execute(text(
            f'CREATE INDEX IF NOT EXISTS idx_silver_{nombre} '
            f'ON silver.{nombre} USING GIST (geometry)'
        ))
    print('  ->', nombre, f'[{operador}]:', len(gdf_proc), 'filas')


## Paso 8 — Ejecutar todo

Por cada operador: consulta la API de CKAN, y si el feed no ha cambiado desde la última carga se omite por completo (ni se descarga ni se sube nada). Si cambió, descarga, construye paradas y rutas, y las sube (reemplazando solo las filas de ese operador). Usa `FORZAR_RECARGA = True` para saltarte la comprobación.


In [10]:
FORZAR_RECARGA = False  # pon en True para recargar aunque la API diga que no hay cambios

paradas_n = 0
rutas_n = 0
hubo_cambios = False

for clave, info in FUENTES_GTFS.items():
    print()
    recurso = obtener_recurso_gtfs(info['package_id'])

    if not FORZAR_RECARGA and not feed_ha_cambiado_geo(info['operador'], recurso['last_modified']):
        print(f"{info['operador']}: sin cambios desde la última carga ({recurso['last_modified']}), se omite.")
        continue

    print('Procesando', info['operador'], '...')
    zf = descargar_gtfs(recurso['url'])
    paradas = construir_paradas(zf, info['operador'], info['modo'])
    rutas = construir_rutas(zf, info['operador'], info['modo'])

    subir_capa(paradas, 'gtfs_paradas', info['operador'])
    subir_capa(rutas, 'gtfs_rutas', info['operador'])

    actualizar_control_carga_geo(info['operador'], recurso['last_modified'])
    paradas_n += len(paradas)
    rutas_n += len(rutas)
    hubo_cambios = True

print()
if hubo_cambios:
    print('Paradas nuevas/actualizadas:', paradas_n)
    print('Lineas nuevas/actualizadas:', rutas_n)
else:
    print('Ningún feed cambió, no se ha subido nada.')



Procesando TITSA ...
  -> gtfs_paradas [TITSA]: 3873 filas
  -> gtfs_rutas [TITSA]: 866 filas

Procesando Metropolitano de Tenerife ...
  -> gtfs_paradas [Metropolitano de Tenerife]: 25 filas
  -> gtfs_rutas [Metropolitano de Tenerife]: 4 filas

Paradas nuevas/actualizadas: 3898
Lineas nuevas/actualizadas: 870


## Paso 9 — Verificar

Contamos cuántas paradas y líneas hay por operador, para confirmar que se cargó tanto TITSA como el tranvía.

In [11]:
with engine.connect() as conn:
    resumen_paradas = pd.read_sql(
        'SELECT operador, modo, COUNT(*) FROM silver.gtfs_paradas GROUP BY operador, modo;', conn
    )
    resumen_rutas = pd.read_sql(
        'SELECT operador, modo, COUNT(*) FROM silver.gtfs_rutas GROUP BY operador, modo;', conn
    )

display(resumen_paradas)
display(resumen_rutas)

,operador,modo,count
0,Metropolitano de Tenerife,tranvia,25
1,TITSA,guagua,3873


,operador,modo,count
0,Metropolitano de Tenerife,tranvia,4
1,TITSA,guagua,866


## Notas

- Las tablas `gtfs_paradas` y `gtfs_rutas` mezclan TITSA y el tranvía en una sola tabla cada una, distinguidos por la columna `operador`. Es una decisión de diseño para tener una única red de transporte público unificada..
- `stop_times.txt` y `calendar.txt` (horarios y días de servicio) no se han cargado aquí porque no son datos espaciales. Los añadiremos más adelante, cuando lleguemos al punto 4.5 del índice (accesibilidad e isocronas), que sí los necesita.
